# ramanujan-qft quickstart

Ramanujan's Master Theorem, Mellin residue summation and arbitrary-precision Feynman integrals in pure Python.

This notebook installs the package and walks through its main functions. Run the cells from top to bottom
(*Runtime → Run all* in Colab). Documentation: [https://reetanshuh1998.github.io/Ramanujan_master_theorem/](https://reetanshuh1998.github.io/Ramanujan_master_theorem/).

*In memory of Srinivasa Ramanujan.*

In [ ]:
%pip install -q "git+https://github.com/reetanshuh1998/Ramanujan_master_theorem"

In [ ]:
import mpmath as mp
import ramanujan_qft
print("ramanujan-qft", ramanujan_qft.__version__)

## 1. Ramanujan's Master Theorem

If $f(x) = \sum_n \varphi(n)(-x)^n/n!$, then $\int_0^\infty x^{s-1} f(x)\,dx = \Gamma(s)\,\varphi(-s)$.
For $(1+x)^{-3}$, $\varphi(n) = \Gamma(3+n)/\Gamma(3)$ and the theorem gives the Beta function $B(s, 3-s)$.
`mellin_numeric` checks it by quadrature.

In [ ]:
from ramanujan_qft import ramanujan_master_theorem, mellin_numeric

phi = lambda n: mp.gamma(3 + n) / mp.gamma(3)
print("RMT       :", ramanujan_master_theorem(phi, 1.25))
print("Beta      :", mp.beta(1.25, 1.75))
print("quadrature:", mellin_numeric(lambda x: (1 + x)**-3, 1.25))

## 2. Product integrals by Mellin residues

`product_integral(f, g, s)` evaluates $\int_0^\infty x^{s-1} f(x)\,g(x)\,dx$ for factors from a small atlas
(exponentials, Gaussians, Bessel $J_\nu$, sines, cosines, inverse powers, $e^{-ax^2}J_\nu(bx)$).
The cost hardly depends on the oscillation frequency, and any number of digits can be requested.

In [ ]:
from ramanujan_qft import product_integral, exponential, bessel_j, cosine, gaussian_bessel

# Int_0^oo exp(-x) J_0(10^5 x) dx = 1/sqrt(1 + 10^10), to 30 digits
value, info = product_integral(exponential(1), bessel_j(0, 1e5), dps=30, return_info=True)
print(mp.nstr(value, 30))
with mp.workdps(30):
    print(mp.nstr(1 / mp.sqrt(1 + mp.mpf(10)**10), 30), "(closed form)")
print(info)

# three-factor product: Int exp(-2x) exp(-x^2/2) J_1(100 x) dx
print(mp.nstr(product_integral(exponential(2), gaussian_bessel("0.5", 100, 1), dps=30), 30))

The engine refuses instead of returning a number it cannot verify, for example at the convergence boundary
$\rho = 1$, or when the integral diverges:

In [ ]:
from ramanujan_qft import SeriesConvergenceError

try:
    product_integral(exponential(1), bessel_j(0, 1))
except SeriesConvergenceError as err:
    print("SeriesConvergenceError:", err)

try:
    product_integral(cosine(1), cosine(2))
except ValueError as err:
    print("ValueError:", err)

## 3. One-loop integrals

`B0` and `C0` follow the LoopTools conventions: squared masses, Minkowski metric ($p^2 > 0$ time-like),
$+i0$ propagators, and $\Delta = 0$, $\mu^2 = 1$ by default. `C0(s, m1sq, m2sq, m3sq)` is
$C_0(0, 0, s; m_1^2, m_2^2, m_3^2)$. Results are precision-checked to the requested number of digits.

In [ ]:
from ramanujan_qft import B0, C0

print("B0(10; 1, 2)     =", B0(10, 1, 2))
print("  LoopTools 2.16:   0.622592073090541875 + 2.01160080643417860 i")
print("C0(0,0,10;1,2,3) =", C0(10, 1, 2, 3))
print("  LoopTools 2.16:  -0.421932531674735434 - 0.336732268033680282 i")
print("C0 to 50 digits  :", mp.nstr(C0(10, 1, 2, 3, dps=50), 50))

## 4. Precision and printing

Results carry the requested precision, but `print` shows them at the global `mp.mp.dps` (15 by default).
Use `mp.nstr(x, n)`. Inputs are exact numbers: a float such as `0.1` is a binary number; pass a string for
the decimal value.

In [ ]:
x = B0(10, 1, 2, dps=40)
print(x)
print(mp.nstr(x, 40))
print(mp.nstr(B0(0.1, 1, 2, dps=30), 30), " (float 0.1)")
print(mp.nstr(B0("0.1", 1, 2, dps=30), 30), " (string '0.1')")

## 5. The two-loop sunset

`sunset_im` is the imaginary part of the massive sunset (zero below the threshold $(m_1+m_2+m_3)^2$).
`sunset_subtracted` is the scheme-independent $S(s) - S(0) - s\,S'(0)$ from a dispersion relation;
it takes 10–20 s per value.

In [ ]:
from ramanujan_qft import sunset_threshold, sunset_im, sunset_subtracted

m = (1, 2, 3)   # squared masses
print("threshold:", sunset_threshold(*m))
for s in (20, 50, 100):
    print(f"Im S({s}) =", mp.nstr(sunset_im(s, *m), 15))
print("S(20) - S(0) - 20 S'(0) =", sunset_subtracted(20, *m))   # about 15 s

## Your turn

Add a cell below and try your own integrals. Next steps:

* [User guide](https://reetanshuh1998.github.io/Ramanujan_master_theorem/user_guide/index.html): each function, its conventions and its refusals
* [API reference](https://reetanshuh1998.github.io/Ramanujan_master_theorem/api/index.html)
* [Validation](https://reetanshuh1998.github.io/Ramanujan_master_theorem/validation.html): what the results are checked against
* [Comparison](https://reetanshuh1998.github.io/Ramanujan_master_theorem/COMPARISON.html): when another tool is the better choice